In [5]:
import pandas as pd
import json
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# -----------------------------
# 1. Simulated Data (structured + JSON logs)
# -----------------------------
data = [
    {"amount": 120.5, "age": 32, "log": '{"device":"mobile","browser":"chrome","os":"android"}', "fraud": 0},
    {"amount": 540.0, "age": 54, "log": '{"device":"desktop","browser":"firefox","os":"windows"}', "fraud": 1},
    {"amount": 75.0, "age": 22, "log": '{"device":"mobile","browser":"safari","os":"ios"}', "fraud": 0},
    {"amount": 999.9, "age": 45, "log": '{"device":"desktop","browser":"chrome","os":"linux"}', "fraud": 1},
]

df = pd.DataFrame(data)

# -----------------------------
# 2. Parse JSON Logs into Dictionary Features
# -----------------------------
df["log_dict"] = df["log"].apply(json.loads)

# -----------------------------
# 3. Convert Categorical JSON Logs into Numeric Features
# -----------------------------
vec = DictVectorizer(sparse=False)
json_features = vec.fit_transform(df["log_dict"])

# Convert back to DataFrame for clarity
df_json = pd.DataFrame(json_features, columns=vec.get_feature_names_out())

# -----------------------------
# 4. Combine Structured + JSON Features
# -----------------------------
X = pd.concat([df[["amount", "age"]], df_json], axis=1)
y = df["fraud"]

# -----------------------------
# 5. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
# -----------------------------
# 6. Train Model
# -----------------------------
#model = RandomForestClassifier(class_weight="balanced", random_state=42)
model = RandomForestClassifier(class_weight="balanced",n_estimators=100, random_state=42)
model.fit(X_res, y_res)

# -----------------------------
# 7. Evaluate
# -----------------------------
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [6]:
from sklearn.metrics import roc_auc_score, confusion_matrix

print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

ROC-AUC: 1.0
Confusion Matrix:
 [[1 0]
 [0 1]]
